# Tutorial 2: Local Pools — Filesystem, Redis, HDF5

Store and recall the same entry across three local backends using a single loop.

This demonstrates LAILA's uniform `memorize` / `remember` / `forget` interface — the code is identical regardless of the storage backend.

In [ ]:
import numpy as np
import laila
from laila.pool import FilesystemPool, RedisPool, HDF5Pool

## Create the pools

Each pool manages its own storage automatically:
- **FilesystemPool** — local disk image
- **RedisPool** — private redis-server on a UNIX socket (no system Redis needed)
- **HDF5Pool** — local `.h5py` file

In [ ]:
pools = [
    ("fs",    FilesystemPool(nickname="tutorial_fs")),
    ("redis", RedisPool(nickname="tutorial_redis")),
    ("hdf5",  HDF5Pool(nickname="tutorial_hdf5")),
]

## Create an entry to store

In [ ]:
entry = laila.constant(data=np.random.randn(10, 10), nickname="tutorial_matrix")
print("Entry global_id:", entry.global_id)
print("Data shape:", entry.data.shape)

## Register, memorize, remember, verify — in a loop

The key point: the code inside the loop is identical for all three backends.

In [ ]:
for nick, pool in pools:
    laila.memory.extend(pool, pool_nickname=nick)

    # Memorize (write)
    future = laila.memorize(entry, pool_nickname=nick)
    laila.wait(future)
    print(f"[{nick}] memorized — status: {laila.status(future)}")

    # Remember (read)
    recall_future = laila.remember(entry.global_id, pool_nickname=nick)
    laila.wait(recall_future)
    recalled = recall_future.result
    print(f"[{nick}] remembered — data shape: {recalled.data.shape}")

    # Verify round-trip
    assert np.array_equal(entry.data, recalled.data), f"Data mismatch on {nick}!"
    print(f"[{nick}] verified")

    # Clean up
    forget_future = laila.forget(entry.global_id, pool_nickname=nick)
    laila.wait(forget_future)
    print(f"[{nick}] cleaned up\n")

## What just happened

1. **`extend`** registered each backend under a nickname
2. **`memorize`** serialized the entry, applied transformations, and wrote to the backend
3. **`wait`** blocked until the write completed
4. **`remember`** fetched the blob, reversed transformations, reconstructed the entry
5. **`forget`** deleted the stored blob

The same steps work for every pool LAILA supports — S3, GCS, Azure, Postgres, MongoDB, and more. Only the pool constructor changes.

**Next:** Tutorial 3 — S3 with NumPy and PyTorch Tensors